
#### Objetivo del notebook

a. Aplicar el estadístico de prueba Chi-cuadrado, con un nivel de confianza del 95%, para establecer si se rechaza la hipótesis nula seguiente:

H0: no hay diferencias significativas entre las características del grupo víctimas previas de VIF-VP y su posteriormente muerte por violencia de esta naturaleza (muerte) y el grupo que experimentó VIF-VP y esta viva (no fatales).

Ha: si hay diferencias significativas entre las características del grupo víctimas previas de VIF-VP y su posteriormente muerte por violencia de esta naturaleza (muerte) y el grupo que experimentó VIF-VP y esta viva (no fatales).

Características:

1. sexo_victima_cod

2. ciclo_vital_cod

3. escolaridad_cod

4. estado_civil_cod

5. dia_del_hecho_cod

6. departamento_del_hecho_dane_cod

7. zona_del_hecho_cod

8. escenario_del_hecho_cod

9. actividad_durante_hecho_cod

10. contexto_del_hecho_cod

11. mecanismo_causal_cod

12. sexo_del_agresor_cod

13. presunto_agresor_cod

-Entrada:

##### Base grupo de personas que experimentaron VIF o VP y estan vivas (no fatales).

VIF_VP_no_fatales_v2

#####Base grupo de personas que fueron víctimas de VIF o VP previamente de su muerte por violencia de esta naturaleza (muerte)

victimas_VIF_VP_con_muerte_homicidio_v2 

-Salida:

b. Obtener y guardar la base de datos 'nofatales_muerte_var_significativas' que contiene la base limpia y con la variable de grupo: nofatales y muerte. Adicionalemente solo con las variables significativas. y 643392 registros.  











In [0]:
#Importar librerías
from pyspark.sql.functions import count, lit, col, when, regexp_replace, concat
from scipy.stats import chi2_contingency
import pandas as pd
from pyspark.sql.types import StringType

from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.stat import ChiSquareTest
import math


**Reagrupar algunas características de estudio**

In [0]:
#Leer tabla de datos como un data frame 1
 
df_VIF_VP_no_fatales_v2= spark.table("ml_proyecto_7405607705157039.default.VIF_VP_no_fatales_v2")

In [0]:
df_VIF_VP_no_fatales_v2.count()

In [0]:
#Leer tabla de datos como un data frame
 
df_victimas_VIF_VP_con_muerte_homicidio_v2= spark.table("ml_proyecto_7405607705157039.default.victimas_VIF_VP_con_muerte_homicidio_v2")

In [0]:
df_victimas_VIF_VP_con_muerte_homicidio_v2.count()

**Apilado**

In [0]:
#crear la variable "grupo" y asignar categoria que identifique las víctimas no fatales y las víctimas fatales de VIF o VP, para luego apilarlas. 

df_VIF_VP_no_fatales_v2_para_apilar = (
    df_VIF_VP_no_fatales_v2.select(
    
    "sexo_victima_cod",
    "ciclo_vital_cod",
    "escolaridad_cod",
    "estado_civil_cod",
    "dia_del_hecho_cod",
    "departamento_del_hecho_dane_cod",
    "zona_del_hecho_cod",
    "escenario_del_hecho_cod",
    "actividad_durante_hecho_cod",
    "contexto_del_hecho_cod",
    "mecanismo_causal_cod",
    "sexo_del_agresor_cod",
    "presunto_agresor_cod",
    "factor_desencadenante_agresion_cod",
    "dias_de_incapacidad_medicolegal_cod"
    )
    .withColumn("grupo", lit("nofatales"))
)

df_victimas_VIF_VP_con_muerte_homicidio_v2_para_apilar = (
    df_victimas_VIF_VP_con_muerte_homicidio_v2.select(

    "sexo_victima_cod",
    "ciclo_vital_cod",
    "escolaridad_cod",
    "estado_civil_cod",
    "dia_del_hecho_cod",
    "departamento_del_hecho_dane_cod",
    "zona_del_hecho_cod",
    "escenario_del_hecho_cod",
    "actividad_durante_hecho_cod",
    "contexto_del_hecho_cod",
    "mecanismo_causal_cod",
    "sexo_del_agresor_cod",
    "presunto_agresor_cod",
    "factor_desencadenante_agresion_cod",
    "dias_de_incapacidad_medicolegal_cod"
    
    )
    .withColumn("grupo", lit("muerte")))

In [0]:
#Apilado.

df_nofatales_muerte_apilado =  df_VIF_VP_no_fatales_v2_para_apilar.unionByName(df_victimas_VIF_VP_con_muerte_homicidio_v2_para_apilar)

In [0]:
df_nofatales_muerte_apilado.count()

In [0]:
df_nofatales_muerte_apilado.printSchema()

In [0]:
#verificar si hay nulos en todo el dataframe

hay_nulos = df_nofatales_muerte_apilado.filter(
    " OR ".join([f"`{c}` IS NULL" for c in df_nofatales_muerte_apilado.columns])
).count()

print(hay_nulos)

In [0]:
#conteo de valores sin información en columnas nuevas

columnas_cod = [
    "sexo_victima_cod",
    "ciclo_vital_cod",
    "escolaridad_cod",
    "estado_civil_cod",
    "dia_del_hecho_cod",
    "departamento_del_hecho_dane_cod",
    "zona_del_hecho_cod",
    "escenario_del_hecho_cod",
    "actividad_durante_hecho_cod",
    "contexto_del_hecho_cod",
    "mecanismo_causal_cod",
    "sexo_del_agresor_cod",
    "presunto_agresor_cod",
    "factor_desencadenante_agresion_cod",
    "dias_de_incapacidad_medicolegal_cod"

]

conteo = df_nofatales_muerte_apilado.select([
    count(
        when(col(c) == "sin informacion", c)
    ).alias(c)
    for c in columnas_cod
])

display(conteo)

**Reagrupar categorías de algunas variables**

In [0]:
#Reagrupar estado_civil_cod

#Valores unicos 
display(
    df_nofatales_muerte_apilado
    .select("estado_civil_cod")
    .distinct()
    .orderBy("estado_civil_cod")
)

In [0]:
#Recodificar valores de la variable estado_civil_cod 

df_nofatales_muerte_apilado = df_nofatales_muerte_apilado.withColumn(
    "estado_civil_cod",
    when(col("estado_civil_cod") == "Unión libre","Unión libre + Casado (a)")
    .when(col("estado_civil_cod") == "Casado (a)","Unión libre + Casado (a)")    
    .otherwise(col("estado_civil_cod"))
)

In [0]:
#valores unicos estado_civil_cod

display(
    df_nofatales_muerte_apilado
    .select("estado_civil_cod")
    .distinct()
    .orderBy("estado_civil_cod")
)

In [0]:
#Reagrupar dia_del_hecho_cod

#Valores unicos 
display(
    df_nofatales_muerte_apilado
    .select("dia_del_hecho_cod")
    .distinct()
    .orderBy("dia_del_hecho_cod")
)

In [0]:
#Recodificar valores de la variable dia_del_hecho_cod

df_nofatales_muerte_apilado = df_nofatales_muerte_apilado.withColumn(
    "dia_del_hecho_cod",
     when(col("dia_del_hecho_cod") == "lunes","Entre semana")
    .when(col("dia_del_hecho_cod") == "martes","Entre semana")
    .when(col("dia_del_hecho_cod") == "miércoles","Entre semana")
    .when(col("dia_del_hecho_cod") == "jueves","Entre semana")    
    .when(col("dia_del_hecho_cod") == "viernes","Entre semana")
    .when(col("dia_del_hecho_cod") == "sábado","Fin de semana")    
    .when(col("dia_del_hecho_cod") == "domingo","Fin de semana")        
    .otherwise(col("dia_del_hecho_cod"))
)

In [0]:
#Valores unicos dia_del_hecho_cod

display(
    df_nofatales_muerte_apilado
    .select("dia_del_hecho_cod")
    .distinct()
    .orderBy("dia_del_hecho_cod")
)

In [0]:
#Reagrupar departamento_del_hecho_dane_cod

#Valores unicos 
display(
    df_nofatales_muerte_apilado
    .select("departamento_del_hecho_dane_cod")
    .distinct()
    .orderBy("departamento_del_hecho_dane_cod")
)

In [0]:
#Quitar comillas en valores
df_nofatales_muerte_apilado1 = df_nofatales_muerte_apilado.withColumn(
    "departamento_del_hecho_dane_cod",
    regexp_replace(col("departamento_del_hecho_dane_cod"), '"', "")
)

In [0]:
#Valores unicos 
display(
    df_nofatales_muerte_apilado
    .select("departamento_del_hecho_dane_cod")
    .distinct()
    .orderBy("departamento_del_hecho_dane_cod")
)

In [0]:
#Recodificar valores de la variable departamento_del_hecho_dane_cod

df_nofatales_muerte_apilado = df_nofatales_muerte_apilado.withColumn(
    "departamento_del_hecho_dane_cod",
     when(col("departamento_del_hecho_dane_cod") == "Amazonas", "Otros departamentos")
    .when(col("departamento_del_hecho_dane_cod") == "Arauca", "Otros departamentos")
    .when(col("departamento_del_hecho_dane_cod") == "Archipiélago de San Andrés, Providencia y Santa Catalina",
                                                    "Otros departamentos")
    .when(col("departamento_del_hecho_dane_cod") == "Atlántico","Otros departamentos")    
    .when(col("departamento_del_hecho_dane_cod") == "Bolívar","Otros departamentos")
    .when(col("departamento_del_hecho_dane_cod") == "Boyacá", "Otros departamentos")
    .when(col("departamento_del_hecho_dane_cod") == "Caldas","Otros departamentos")
    .when(col("departamento_del_hecho_dane_cod") == "Caquetá","Otros departamentos")    
    .when(col("departamento_del_hecho_dane_cod") == "Casanare","Otros departamentos")
    .when(col("departamento_del_hecho_dane_cod") == "Cauca", "Otros departamentos")
    .when(col("departamento_del_hecho_dane_cod") == "Cesar","Otros departamentos")
    .when(col("departamento_del_hecho_dane_cod") == "Chocó","Otros departamentos")    
    .when(col("departamento_del_hecho_dane_cod") == "Córdoba","Otros departamentos")
    .when(col("departamento_del_hecho_dane_cod") == "Guainía", "Otros departamentos")
    .when(col("departamento_del_hecho_dane_cod") == "Guaviare","Otros departamentos")
    .when(col("departamento_del_hecho_dane_cod") == "Huila","Otros departamentos")    
    .when(col("departamento_del_hecho_dane_cod") == "La Guajira","Otros departamentos")
    .when(col("departamento_del_hecho_dane_cod") == "Magdalena","Otros departamentos")
    .when(col("departamento_del_hecho_dane_cod") == "Meta","Otros departamentos")    
    .when(col("departamento_del_hecho_dane_cod") == "Nariño","Otros departamentos") 
    .when(col("departamento_del_hecho_dane_cod") == "Norte de Santander","Otros departamentos")
    .when(col("departamento_del_hecho_dane_cod") == "Putumayo","Otros departamentos")    
    .when(col("departamento_del_hecho_dane_cod") == "Quindio","Otros departamentos")
    .when(col("departamento_del_hecho_dane_cod") == "Risaralda","Otros departamentos") 
    .when(col("departamento_del_hecho_dane_cod") == "Santander","Otros departamentos")
    .when(col("departamento_del_hecho_dane_cod") == "Sucre","Otros departamentos")    
    .when(col("departamento_del_hecho_dane_cod") == "Tolima","Otros departamentos")
    .when(col("departamento_del_hecho_dane_cod") == "Vaupés","Otros departamentos")
    .when(col("departamento_del_hecho_dane_cod") == "Vichada","Otros departamentos")                   
    .otherwise(col("departamento_del_hecho_dane_cod"))
)

In [0]:
#Valores unicos 
display(
    df_nofatales_muerte_apilado
    .select("departamento_del_hecho_dane_cod")
    .distinct()
    .orderBy("departamento_del_hecho_dane_cod")
)

In [0]:
#Reagrupar escenario_del_hecho_cod

#Valores unicos 
display(
    df_nofatales_muerte_apilado
    .select("escenario_del_hecho_cod")
    .distinct()
    .orderBy("escenario_del_hecho_cod")
)

In [0]:
#Quitar comillas en valores
df_nofatales_muerte_apilado = df_nofatales_muerte_apilado.withColumn(
    "escenario_del_hecho_cod",
    regexp_replace(col("escenario_del_hecho_cod"), '"', "")
)

In [0]:
#Recodificar valores de la variable escenario_del_hecho_cod

df_nofatales_muerte_apilado = df_nofatales_muerte_apilado.withColumn(
    "escenario_del_hecho_cod",
     when(col("escenario_del_hecho_cod") == "Ambulancia - transporte sanitario", "Fuera de la vivienda")
     .when(col("escenario_del_hecho_cod") == "Calle (autopista, avenida, dentro de la ciudad)", "Fuera de la vivienda")     
    .when(col("escenario_del_hecho_cod") == "Carretera (fuera de la ciudad)", "Fuera de la vivienda")
    .when(col("escenario_del_hecho_cod") == "Centro de atención médica (hospital, clínica, consultorio, etc.)","Fuera de la vivienda")
    .when(col("escenario_del_hecho_cod") == "Centros de reclusión", "Fuera de la vivienda")
    .when(col("escenario_del_hecho_cod") == "Centros educativos","Fuera de la vivienda")
    .when(col("escenario_del_hecho_cod") == "Espacios acuáticos al aire libre (mar, rio, arroyo, humedal, lago, etc.)", "Fuera de la vivienda")
    .when(col("escenario_del_hecho_cod") == "Espacios terrestres al aire libre (bosque, potrero, montaña, playa, etc.)","Fuera de la vivienda")
    .when(col("escenario_del_hecho_cod") == "Establecimiento comercial (tienda, centro comercial, almacén)", "Fuera de la vivienda")
    .when(col("escenario_del_hecho_cod") == "Establecimiento comercial (tienda, centro comercial, almacén, plaza de mercado)","Fuera de la vivienda")
    .when(col("escenario_del_hecho_cod") == "Establecimiento industrial (fábrica, planta) y/o obras en construcción", "Fuera de la vivienda")
    .when(col("escenario_del_hecho_cod") == "Establecimientos Financieros y Relacionados (Bancos,Fiduciarias,Etc) ","Fuera de la vivienda")    
    .when(col("escenario_del_hecho_cod") == "Establecimientos de expendio de comidas (restaurantes, asaderos, salsamentarias, etc.)", "Fuera de la vivienda")
    .when(col("escenario_del_hecho_cod") == "Establecimientos dedicados a la administración pública (cortes, juzgados, ministerios, etc.)","Fuera de la vivienda")
    .when(col("escenario_del_hecho_cod") == "Estaciones de servicio (bombas de gasolina)", "Fuera de la vivienda")
    .when(col("escenario_del_hecho_cod") == "Guarniciones militares y/o de policía","Fuera de la vivienda")
    .when(col("escenario_del_hecho_cod") == "Lugar de explotación de minas y canteras", "Fuera de la vivienda")
    .when(col("escenario_del_hecho_cod") == "Lugares de Hospedaje (Hoteles,Campamentos y Otros Tipos de Hospedaje No Permanente,Moteles,Etc)","Fuera de la vivienda")  
    .when(col("escenario_del_hecho_cod") == "Lugares de actividades culturales (cines, teatros, museos, bibliotecas, etc.)", "Fuera de la vivienda")
    .when(col("escenario_del_hecho_cod") == "Lugares de cuidado de personas (hospicios, orfelinatos, hogares geriátricos, etc.)","Fuera de la vivienda")
    .when(col("escenario_del_hecho_cod") == "Lugares de esparcimiento con expendio de alcohol", "Fuera de la vivienda")
    .when(col("escenario_del_hecho_cod") == "Medio de transporte masivo","Fuera de la vivienda")
    .when(col("escenario_del_hecho_cod") == "Oficinas y/o edificios de oficinas", "Fuera de la vivienda")
    .when(col("escenario_del_hecho_cod") == "Otros","Fuera de la vivienda")
    .when(col("escenario_del_hecho_cod") == "Parqueaderos, estacionamientos","Fuera de la vivienda")  
    .when(col("escenario_del_hecho_cod") == "Piscina y jacuzzi (establecimientos turísticos, recreativos, deportivos)", "Fuera de la vivienda")
    .when(col("escenario_del_hecho_cod") == "Piscina y jacuzzi (vivienda)","Vivienda")
    .when(col("escenario_del_hecho_cod") == "Sitio de culto (capilla, iglesia, templo, etc.)", "Fuera de la vivienda")
    .when(col("escenario_del_hecho_cod") == "Taller","Fuera de la vivienda")
    .when(col("escenario_del_hecho_cod") == "Terminales de pasajeros", "Fuera de la vivienda")
    .when(col("escenario_del_hecho_cod") == "Terreno baldío","Fuera de la vivienda")
    .when(col("escenario_del_hecho_cod") == "Vehículo de servicio particular","Fuera de la vivienda")  
    .when(col("escenario_del_hecho_cod") == "Vehículo de transporte", "Fuera de la vivienda")
    .when(col("escenario_del_hecho_cod") == "Vía pública","Fuera de la vivienda")
    .when(col("escenario_del_hecho_cod") == "Zonas de actividades agropecuarias", "Fuera de la vivienda")
    .when(col("escenario_del_hecho_cod") == "Áreas deportivas y/o recreativas","Fuera de la vivienda")          
    .otherwise(col("escenario_del_hecho_cod"))
)

In [0]:
#Valores unicos 
display(
    df_nofatales_muerte_apilado
    .select("escenario_del_hecho_cod")
    .distinct()
    .orderBy("escenario_del_hecho_cod")
)

In [0]:
#Reagrupar mecanismo_causal_cod

#Valores unicos 
display(
    df_nofatales_muerte_apilado
    .select("mecanismo_causal_cod")
    .distinct()
    .orderBy("mecanismo_causal_cod")
)

In [0]:
#Recodificar valores de la variable escenario_del_hecho_cod

df_nofatales_muerte_apilado = df_nofatales_muerte_apilado.withColumn(
    "mecanismo_causal_cod",
     when(col("mecanismo_causal_cod") == "Abrasivo", "Otros mecanismos")
     .when(col("mecanismo_causal_cod") == "Agente químico", "Otros mecanismos")
     .when(col("mecanismo_causal_cod") == "Agente químico corrosivo","Otros mecanismos")
     .when(col("mecanismo_causal_cod") == "Agente químico irritante","Otros mecanismos")
     .when(col("mecanismo_causal_cod") == "Agente químico tóxico","Otros mecanismos")
     .when(col("mecanismo_causal_cod") == "Agentes y mecanismo explosivo","Otros mecanismos")
     .when(col("mecanismo_causal_cod") == "Agentes y mecanismos biológicos","Otros mecanismos")
     .when(col("mecanismo_causal_cod") == "Biodinámico","Otros mecanismos")
     .when(col("mecanismo_causal_cod") == "Caústico","Otros mecanismos")
     .when(col("mecanismo_causal_cod") == "Cortante","Otros mecanismos")
     .when(col("mecanismo_causal_cod") == "Corto contundente","Otros mecanismos")
     .when(col("mecanismo_causal_cod") == "Corto punzante","Otros mecanismos")
     .when(col("mecanismo_causal_cod") == "Eléctrico","Otros mecanismos")
     .when(col("mecanismo_causal_cod") == "Generadores de asfixia","Otros mecanismos")
     .when(col("mecanismo_causal_cod") == "Proyectil de arma de fuego","Otros mecanismos")
     .when(col("mecanismo_causal_cod") == "Punzante","Otros mecanismos")
     .when(col("mecanismo_causal_cod") == "Térmico","Otros mecanismos")
     .when(col("mecanismo_causal_cod") == "Tóxico","Otros mecanismos")            
    .otherwise(col("mecanismo_causal_cod"))
)

In [0]:
#Valores unicos 
display(
    df_nofatales_muerte_apilado
    .select("mecanismo_causal_cod")
    .distinct()
    .orderBy("mecanismo_causal_cod")
)

In [0]:
#Reagrupar presunto_agresor_cod

#Valores unicos 
display(
    df_nofatales_muerte_apilado
    .select("presunto_agresor_cod")
    .distinct()
    .orderBy("presunto_agresor_cod")
)

In [0]:
#Recodificar valores de la variable presunto_agresor_cod

df_nofatales_muerte_apilado = df_nofatales_muerte_apilado.withColumn(
    "presunto_agresor_cod",
     when(col("presunto_agresor_cod") == "Compañero(a) permanente", "Compañero(a) permanente -Esposo(a)")
     .when(col("presunto_agresor_cod") == "Esposo(a)","Compañero(a) permanente -Esposo(a)")
     .when(col("presunto_agresor_cod") == "Ex compañero(a) permanente","Ex compañero(a) permanente")
     .when(col("presunto_agresor_cod") == "Ex compañero(a) sentimental","Ex compañero(a) permanente")
     .when(col("presunto_agresor_cod") == "Ex esposo(a)","Ex compañero(a) permanente")
     .when(col("presunto_agresor_cod") == "Ex novio(a)","Ex compañero(a) permanente")
     .when(col("presunto_agresor_cod") == "Padre","Padre-Padrastro")
     .when(col("presunto_agresor_cod") == "Padrastro","Padre-Padrastro")
     .when(col("presunto_agresor_cod") == "Madre","Madre-Madrastra")
     .when(col("presunto_agresor_cod") == "Madrastra","Madre-Madrastra")
     .when(col("presunto_agresor_cod") == "Abuelo(a)","Otros familiares civiles o consanguíneos")
     .when(col("presunto_agresor_cod") == "Encargado del cuidado","Otros familiares civiles o consanguíneos")
     .when(col("presunto_agresor_cod") == "Nieto(a)","Otros familiares civiles o consanguíneos") 
     .when(col("presunto_agresor_cod") == "Novio(a)","Otros familiares civiles o consanguíneos")
     .when(col("presunto_agresor_cod") == "Nuera","Otros familiares civiles o consanguíneos")
     .when(col("presunto_agresor_cod") == "Pareja o ex pareja","Otros familiares civiles o consanguíneos")
     .when(col("presunto_agresor_cod") == "Personal de custodia","Otros familiares civiles o consanguíneos")
     .when(col("presunto_agresor_cod") == "Primo(a)","Otros familiares civiles o consanguíneos")
     .when(col("presunto_agresor_cod") == "Profesor (a)","Otros familiares civiles o consanguíneos") 
     .when(col("presunto_agresor_cod") == "Sobrino(a)","Otros familiares civiles o consanguíneos")
     .when(col("presunto_agresor_cod") == "Suegro(a)","Otros familiares civiles o consanguíneos")
     .when(col("presunto_agresor_cod") == "Yerno","Otros familiares civiles o consanguíneos")
     .when(col("presunto_agresor_cod") == "Amante","Otros familiares civiles o consanguíneos")
     .when(col("presunto_agresor_cod") == "Ex amante","Otros familiares civiles o consanguíneos")
    .otherwise(col("presunto_agresor_cod"))
)

In [0]:
#Valores unicos 
display(
    df_nofatales_muerte_apilado
    .select("presunto_agresor_cod")
    .distinct()
    .orderBy("presunto_agresor_cod")
)

In [0]:
df_nofatales_muerte_apilado.printSchema()

In [0]:
#aplicar Chi-cuadrado

variables = [

    "sexo_victima_cod",
    "ciclo_vital_cod",
    "escolaridad_cod",
    "estado_civil_cod",
    "dia_del_hecho_cod",
    "departamento_del_hecho_dane_cod",
    "zona_del_hecho_cod",
    "escenario_del_hecho_cod",
    "actividad_durante_hecho_cod",
    "contexto_del_hecho_cod",
    "mecanismo_causal_cod",
    "sexo_del_agresor_cod",
    "presunto_agresor_cod",
   
]

resultados = []

for variable in variables:

    tabla = (
        df_nofatales_muerte_apilado.groupBy(variable)
          .pivot("grupo")
          .count()
          .fillna(0)
          .orderBy(variable)
          .toPandas()
    )

    # La primera columna contiene las categorías
    matriz = tabla.iloc[:, 1:].values

    chi2, p, gl, esperado = chi2_contingency(matriz)

    resultados.append({
        "Variable": variable,
        "Chi2": round(chi2, 3),
        "gl": gl,
        "p_valor": round(p, 5),
        "Diferencia": "Sí" if p < 0.05 else "No"
    })

In [0]:
resumen = pd.DataFrame(resultados)

print(resumen)

Interpretación: con un nivel de confianza del 95% se rechaza la hipótesis nula. Existen diferencias significativas entre los grupos no fatales y muerte en 7 características: sexo de la víctima, ciclo vital, escolaridad, estado civil, contexto del hecho, mecanismo causal y presunto agresor. 

In [0]:
#Guardar base de datos que contiene la base limpia y con la variable grupo: nofatales y muerte

df_nofatales_muerte_apilado.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(
        "ml_proyecto_7405607705157039.default.nofatales_muerte_apilado"
    )

In [0]:
columnas = [
    "sexo_victima_cod",
    "ciclo_vital_cod",
    "escolaridad_cod",
    "estado_civil_cod",
    "contexto_del_hecho_cod",
    "mecanismo_causal_cod",
    "presunto_agresor_cod",
    "grupo"
]

df_nofatales_muerte_var_significativas = df_nofatales_muerte_apilado.select(columnas)

In [0]:
df_nofatales_muerte_var_significativas.printSchema()

In [0]:
#Guardar base de datos que contiene la base limpia y con la variable grupo: nofatales y muerte. Adicionalemente solo las variables significativas

df_nofatales_muerte_var_significativas.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(
        "ml_proyecto_7405607705157039.default.nofatales_muerte_var_significativas"
    )

In [0]:
df_nofatales_muerte_var_significativas.printSchema()